# Per-SRR metadata

Fetches SRA metadata for the 253 selected SRRs (`dataset_discovery/samples.tsv`)
and cross-checks it against the final curated 510-contig / 131-SRR set from
`filtering_snakemake_contigs.ipynb`, plus the domain-expert category
annotation.

The pident 50-95% selection itself is derived once in
`dataset_discovery/filter_results.ipynb`; this notebook just loads its output
(`samples.tsv`) rather than recomputing it.

`results/metadata.csv` is the same 253-row metadata later published as
`Supplementary_table4.tsv` in `tobamo-supp-data` (up to tsv/csv formatting).

In [6]:
import pandas as pd
from Bio import SeqIO
from pysradb.sraweb import SRAweb

## Load the 253-SRR selection and the final curated set

In [7]:
samples_path = '../../dataset_discovery/samples.tsv'
with open(samples_path) as file:
    samples = [line.strip() for line in file.readlines()][1:]

non_cellular_path = '../../data/contigs/contigs_non_cellular_filtered.fasta'
records = list(SeqIO.parse(non_cellular_path, 'fasta'))
record_names = sorted({r.id.split('_')[-1] for r in records})

# every curated SRR should trace back to the 253 selection
assert set(record_names).issubset(set(samples))

print(f"253-SRR selection: {len(samples)} SRRs")
print(f"Final curated set: {len(record_names)} SRRs, {len(records)} contigs")

253-SRR selection: 253 SRRs
Final curated set: 131 SRRs, 510 contigs


## Fetch metadata for the 253 selected SRRs

In [8]:
# RUN ONLY ONCE -- hits the SRA API
# db = SRAweb()
# metadata = db.sra_metadata(samples, detailed=True)
# metadata.to_csv('../results/metadata.csv')

metadata = pd.read_csv('../results/metadata.csv', index_col=0)

## Flag SRRs in the final curated set

In [9]:
selected_metadata = metadata.copy()
selected_metadata['contigs_510'] = selected_metadata['run_accession'].isin(record_names).astype(int)

selected_metadata.to_csv('../results/selected_metadata.csv')